# Matched 8D and 7D simulation banks

This notebook audits the immutable, paired simulation banks used by the new
Figure-10-equivalent experiment. The 8D and 7D rows share the first seven
Sobol coordinates and simulator-seed schedule; only `c_ctx2th` is inferred in
8D and fixed in 7D. The signal is a **synthetic cortical-rate observable**, not
scalp EEG.

In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "S4_sbi").exists() else cwd.parent.parent
SRC = PROJECT_ROOT / "S4_sbi" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
RESULTS = PROJECT_ROOT / "S4_sbi" / "results" / "figure10_8d_7d"
assert os.environ.get("CONDA_DEFAULT_ENV") == "neurolib"
print("environment:", os.environ.get("CONDA_DEFAULT_ENV"))
print("python:", sys.executable)
print("results:", RESULTS)

environment: neurolib
python: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib\python.exe
results: D:\Year3_Mao_Projects\sleep_loop\S4_sbi\results\figure10_8d_7d


## Resource decision and checkpoint evidence

The official three-million scale could not be completed on this CPU within a
bounded reproducible run. The preregistered final scale is therefore 32,768
valid rows per track, supported by a measured throughput projection.

In [2]:
benchmark = json.loads((RESULTS / "resource_benchmark" / "resource_benchmark.json").read_text())
manifest = json.loads((RESULTS / "matched_banks" / "matched_bank_manifest.json").read_text())
display(pd.DataFrame(benchmark["projections"]).T)
display(pd.Series(manifest, name="matched bank manifest"))
assert manifest["8d_valid"] == manifest["target_per_track"]
assert manifest["7d_valid"] == manifest["target_per_track"]
assert manifest["paired_valid"] == manifest["target_per_track"]

,paired_tracks_wall_hours,paired_tracks_wall_days,raw_array_gib_estimate
8192,4.809296,0.200387,0.003174
32768,19.237182,0.801549,0.012695
131072,76.948729,3.206197,0.050781
524288,307.794918,12.824788,0.203125
1000000,587.072215,24.461342,0.387430
3000000,1761.216646,73.384027,1.162291


created_utc                                2026-07-29T15:46:13.881061+00:00
target_per_track                                                      32768
8d_valid                                                              32768
8d_failed                                                                 0
8d_failure_rate                                                         0.0
7d_valid                                                              32768
7d_failed                                                                 0
7d_failure_rate                                                         0.0
paired_valid                                                          32768
train_rows                                                            26214
validation_rows                                                        6554
bank_paths                {'8d': 'matched_banks/8d/figure10_8d_bank_3276...
bank_sha256               {'8d': '75f4ebcf785db5c866509e63657d7a016c5089...
split_path  

## Contract and paired-row checks

These assertions reject silent parameter-order drift, nonfinite summaries,
object arrays, unmatched shared coordinates, and reuse of a fixed nuisance
seed.

In [3]:
bank8_path = RESULTS / manifest["bank_paths"]["8d"]
bank7_path = RESULTS / manifest["bank_paths"]["7d"]
with np.load(bank8_path, allow_pickle=False) as b8, np.load(bank7_path, allow_pickle=False) as b7:
    audit = {
        "8D theta": b8["theta"].shape,
        "7D theta": b7["theta"].shape,
        "8D x": b8["x"].shape,
        "7D x": b7["x"].shape,
        "8D finite": bool(np.isfinite(b8["x"]).all()),
        "7D finite": bool(np.isfinite(b7["x"]).all()),
        "shared theta exact": bool(np.array_equal(b8["theta"][:, :7], b7["theta"])),
        "seed schedule exact": bool(np.array_equal(b8["simulator_seed"], b7["simulator_seed"])),
        "distinct simulator seeds": int(np.unique(b8["simulator_seed"]).size),
        "feature order match": bool(np.array_equal(b8["feature_names"], b7["feature_names"])),
        "object arrays": [key for key in b8.files if b8[key].dtype == object] + [key for key in b7.files if b7[key].dtype == object],
    }
display(pd.Series(audit))
assert all([audit["8D finite"], audit["7D finite"], audit["shared theta exact"], audit["seed schedule exact"], audit["feature order match"]])
assert not audit["object arrays"]

8D theta                     (32768, 8)
7D theta                     (32768, 7)
8D x                        (32768, 14)
7D x                        (32768, 14)
8D finite                          True
7D finite                          True
shared theta exact                 True
seed schedule exact                True
distinct simulator seeds          32768
feature order match                True
object arrays                        []
dtype: object

## Interpretation

This establishes a matched engineering comparison. It does not establish
parameter identifiability, posterior calibration, or a source-to-Fpz-Cz
measurement model. Those claims require the later diagnostics.